# Advanced Context Retrieval

Today's goal is to take the chunk context that we have in our database and implement advanced context retrieval methods to improve our answers. This will involve learning the methods (Notion notes) and them implementing them properly in our codebase.

NOTE: All of this code will be in retrieval_engine.py

In [ ]:
import sqlite3
import ollama
import pandas as pd 
from sentence_transformers import SentenceTransformer
import chromadb
from pathlib import Path

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

root=Path(r"C:\Users\micro\Documents\ABTalksAI-Cohort")
jsonroot = root / "knowledge_base_embed.jsonl"
db_path = root / "chroma_db"
dpath = root / "data"


dfp = pd.read_csv(dpath + "plans.csv")
dfc = pd.read_csv(dpath + "claims.csv")

conn = sqlite3.connect(dpath + "coverage.db")
dfp.to_sql("plans", conn, if_exists="replace", index=False)
dfc.to_sql("claims", conn, if_exists="replace", index=False)
conn.commit()

client = chromadb.PersistentClient(path=db_path)
collection = client.get_collection(name="coverage_kb")

In [ ]:
# write a function that classifies our question using simple keyword classifiers
def define_function(question):
    question = question.lower()

    structured_keywords = [
        "deductible", "premium", "claim status", "claim amount",
        "claim id", "member id", "plan id", "date filed", "copay"
    ]

    unstructured_keywords = [
        "covered", "coverage", "procedure", "prior authorization",
        "appeal", "in-network", "out-of-network", "excluded", "eligible"
    ]

    structured_match = any(word in question for word in structured_keywords)
    unstructured_match = any(word in question for word in unstructured_keywords)

    if structured_match and unstructured_match:
        return "both"
    elif structured_match:
        return "structured"
    else:
        return "unstructured"



def sql_lookup(question, structure):
    if structure not in ("both", "structured"):
        return None
    
    user = f"""
    Convert the following question to a raw SQL query for structured RAG retrieval.

    Question: {question}

    Database schema:
    plans: plan_id, plan_name, monthly_premium, annual_deductible, copay_pct, coverage_type, network_tier
    claims: claim_id, member_id, plan_id, procedure, claim_amount, status, date_filed

    Return only the SQL query.
    """

    reply = ollama.chat(
        model="qwen2.5-coder:14b",
        messages=[
            {"role": "user", "content": user}
        ]
    )

    text = reply["message"]["content"]
    cursor = conn.cursor()

    no_fly_list = ["```", "delete", "create", "here", "sql", "update", "drop", "insert", "alter"]
    if not any(word in text.lower() for word in no_fly_list):
        cursor.execute(text)

    rows = cursor.fetchall()

    return rows



next, create the function for unstuctured queries- we can effectively just copy our old code

In [ ]:

def vector_lookup(question, structure): 
    if structure not in ("unstructured"):
        return None

    emb_query = model.encode(question)
    results = collection.query(
        query_embeddings=[emb_query],
        n_results=5,
    )
    return results


In [ ]:
def merge_context(rows, results):
    context_items = []

    for row in rows or []:
        context_items.append(f"Structured database result:\n{row}")

    documents = results.get("documents", [[]])[0] if results else []

    for document in documents:
        context_items.append(f"Retrieved document:\n{document}")

    seen = set()
    unique_items = []

    for item in context_items:
        normalized = " ".join(item.lower().split())

        if normalized not in seen:
            seen.add(normalized)
            unique_items.append(item)

    return "\n\n---\n\n".join(unique_items)


"""
I used AI for this block, it:

Converts every SQL row into a labeled text block.
Extracts Chroma’s returned documents.
Normalizes whitespace and capitalization for comparison.
Removes exact duplicate blocks while preserving order.
Keeps structured and unstructured evidence visibly separated.
Joins everything into one prompt-ready context string.

"""

In [ ]:
# call all the functions:
def retrieve(question):
    structure = define_function(question)
    rows = sql_lookup(question, structure) or []
    results = vector_lookup(question, structure) or []
    model_context = merge_context(rows, results)

    return model_context

In [ ]:
test_questions = [
    "What's my copay percentage?",
    "What is the annual deductible for Silver HMO?",
    "What is the monthly premium for Gold PPO?",
    "What is the claim status for C1003?",
    "What is the claim amount for C1002?",
    "Is maternity care covered on the Bronze HMO plan?",
    "Do I need prior authorization for surgery?",
    "Can I use an out-of-network doctor?",
    "How do I appeal a denied claim?",
    "What is the claim status for C1003, and how do I appeal if it was denied?"
]

for question in test_questions:
    print("=" * 80)
    print("Question:", question)
    print("Structure:", define_function(question))
    print("Result:", retrieve(question))

# I also had AI generate these questions to keep things simple.